In [1]:
#!git clone https://github.com/whyhardt/SPICE.git

In [2]:
# !pip install -e SPICE

In [3]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from spice import SpiceEstimator, SpiceConfig, csv_to_dataset, BaseModel, plot_session, split_data_along_sessiondim

sys.path.append('../../..')
from weinhardt2026.utils.benchmarking_gru import GRUModel, training
from benchmarking_braun2018 import ExpectedValueControl, get_dataset, generate_behavior
from spice_braun2018 import SpiceModel, CONFIG

# For custom RNN
import torch
import torch.nn as nn

# NOTEBOOK CONFIG

In [4]:
train_spice = True
train_evc = False
train_gru = False

# DATASET

## Load dataset

In [5]:
dataset_train, dataset_test, _ = get_dataset(verbose=True)

Shape of dataset: torch.Size([756, 274, 1, 10])
Number of participants: 63
Number of actions in dataset: 2


## Dataset processing

In [6]:
# seems like there are a few blocks that are way longer than the average block
# maybe limiting the block length to a given size would help
n_trials_per_block = np.zeros(dataset_train.xs.shape[0])
for index, block in enumerate(dataset_train.xs[:, :, 0, 0]):
    n_trials_per_block[index] = block.shape[0]-block.isnan().sum()

mean, std = n_trials_per_block.mean(), n_trials_per_block.std()

print(f"Average sequence length: {int(mean)}+-{int(std)}")
print(f"Old max sequence length is {dataset_train.xs.shape[1]}")

# it would be okay to limit until ca mean+2*std trials per  (~200 trials)
dataset_train.xs = dataset_train.xs[:, :int(mean+2*std)]
dataset_train.ys = dataset_train.ys[:, :int(mean+2*std)]

print(f"New max sequence length is {dataset_train.xs.shape[1]}")

Average sequence length: 66+-39
Old max sequence length is 274
New max sequence length is 145


In [7]:
# seems like there are a few blocks that are way longer than the average block
# maybe limiting the block length to a given size would help
n_trials_per_block = np.zeros(dataset_test.xs.shape[0])
for index, block in enumerate(dataset_test.xs[:, :, 0, 0]):
    n_trials_per_block[index] = block.shape[0]-block.isnan().sum()

mean, std = n_trials_per_block.mean(), n_trials_per_block.std()

print(f"Average sequence length: {int(mean)}+-{int(std)}")
print(f"Old max sequence length is {dataset_test.xs.shape[1]}")

# it would be okay to limit until ca mean+2*std trials per  (~200 trials)
dataset_test.xs = dataset_test.xs[:, :int(mean+2*std)]
dataset_test.ys = dataset_test.ys[:, :int(mean+2*std)]

print(f"New max sequence length is {dataset_test.xs.shape[1]}")

Average sequence length: 67+-34
Old max sequence length is 274
New max sequence length is 136


# SPICE

## SPICE Setup

Let's setup now the `SpiceEstimator` object and fit it to the data!

In [8]:
path_spice = 'params/spice_braun2018.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=SpiceModel,
        spice_config=CONFIG,
        n_actions=2,
        n_participants=dataset_train.n_participants,
        n_experiments=1,
        n_reward_features=0,
        
        epochs=1000,
        warmup_steps=500,

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        verbose=True,
        save_path_spice=path_spice,
    )

In [9]:
if train_spice:
    estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

100%|█████████████████████████| 1000/1000 [33:16<00:00,  2.00s/it, L(Train)=0.4259250, L(Val,RNN)=0.4387146, L(Val,SINDy)=0.4711224, Conv=3.71e-08, LR=1.00e-05]
----------------------------------------------------------------------------------------------------------------------------------------------------------------
SPICE Model (Coefficients: 23):
reward_repeat[t+1]  = 0.519 1 + 1.102 dreward_tasks + -0.022 dreward_tasks^2 
reward_switch[t+1]  = -0.463 1 + 1.131 dreward_tasks + 0.013 dreward_tasks^2 
task_repeat[t+1]    = -0.136 1 + 0.895 task_repeat[t] + 0.08 repeat + -0.014 task_repeat^2 + 0.059 task_repeat*repeat + 0.079 repeat^2 
task_switch[t+1]    = 0.103 1 + 0.944 task_switch[t] + -0.099 repeat + 0.025 task_switch*repeat + -0.099 repeat^2 
fatigue_repeat[t+1] = 0.198 1 + 0.011 block + 0.064 block^2 
fatigue_switch[t+1] = -0.195 1 + -0.037 block + -0.038 block^2 
------------------------------------------------------------------------------------------------------------------

 10%|█         | 102/1000 [00:08<01:11, 12.59it/s, loss=0.0072576, lr=1.0e-02, n_params=24.00+/-0.00]

Ensemble confidence filtering:
	reward_repeat: 189 -> 24 / 189 (participant, experiment, term) slots
	reward_switch: 189 -> 17 / 189 (participant, experiment, term) slots
	task_repeat: 378 -> 360 / 378 (participant, experiment, term) slots
	task_switch: 378 -> 356 / 378 (participant, experiment, term) slots
	fatigue_repeat: 189 -> 0 / 189 (participant, experiment, term) slots
	fatigue_switch: 189 -> 0 / 189 (participant, experiment, term) slots


 20%|██        | 202/1000 [00:16<01:03, 12.48it/s, loss=0.0072288, lr=1.0e-02, n_params=19.00+/-0.00]

Ensemble confidence filtering:
	reward_repeat: 189 -> 9 / 189 (participant, experiment, term) slots
	reward_switch: 189 -> 9 / 189 (participant, experiment, term) slots
	task_repeat: 378 -> 360 / 378 (participant, experiment, term) slots
	task_switch: 378 -> 354 / 378 (participant, experiment, term) slots
	fatigue_repeat: 189 -> 0 / 189 (participant, experiment, term) slots
	fatigue_switch: 189 -> 0 / 189 (participant, experiment, term) slots


...

Ensemble confidence filtering:
	reward_repeat: 0 -> 0 / 189 (participant, experiment, term) slots
	reward_switch: 0 -> 0 / 189 (participant, experiment, term) slots
	task_repeat: 357 -> 355 / 378 (participant, experiment, term) slots
	task_switch: 342 -> 339 / 378 (participant, experiment, term) slots
	fatigue_repeat: 0 -> 0 / 189 (participant, experiment, term) slots
	fatigue_switch: 0 -> 0 / 189 (participant, experiment, term) slots


100%|██████████| 1000/1000 [01:27<00:00, 11.43it/s, loss=0.0072060, lr=1.0e-05, n_params=11.10+/-0.80]



Stage 2.2: SINDy coefficient estimation (multi-step shooting, K=100)
Re-computing state trajectories on full data (per-member targets)...
Ridge regression succeeded (K=100 loss: 0.2884768). Running SGD refinement...


 54%|█████▍    | 541/1000 [06:13<05:16,  1.45it/s, K=100, loss=0.2746365, lr=1.0e-05, n_params=11.10+/-0.80]



Stage 2.2 interrupted. Continuing...

Losses:
	         Training    Validation
	RNN      0.41778     0.43871
	SINDy    0.59047     0.60428

RNN training finished.
Training took 2461.76 seconds.
Saving SPICE model to params/spice_braun2018.pkl...


In [ ]:
estimator.load_spice(path_spice)

# BENCHMARKING

### Expected Value of Control Model (Shenhav et al., 2013)

In [10]:
path_spice = 'params/spice_braun2018.pkl'

evc = ExpectedValueControl(n_participants=dataset_train.n_participants)
path_evc = path_spice.replace('spice_', 'evc_')

In [11]:
if train_evc:
    optimizer_evc = torch.optim.Adam(evc.parameters(), lr=0.01)
    epochs = 1000

    evc = training(
        model=evc,
        optimizer=optimizer_evc,
        dataset_train=dataset_train,
        dataset_test=dataset_test,
        epochs=epochs,
        device=torch.device('cpu'),
    )

    torch.save(evc.state_dict(), path_evc)

In [12]:
evc.load_state_dict(torch.load(path_evc, map_location='cpu'))

<All keys matched successfully>

### GRU Model

In [13]:
gru = GRUModel(dataset_train.n_actions, additional_inputs=4, n_reward_features=0)
path_gru = path_spice.replace('spice_', 'gru_')

In [14]:
if train_gru:
    epochs = 1000
    optimizer_gru = torch.optim.Adam(gru.parameters(), lr=0.01)

    gru = training(
        model=gru.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu')),
        optimizer=optimizer_gru,
        dataset_train=dataset_train,
        dataset_test=dataset_test,
        epochs=epochs,
        ).to(torch.device('cpu'))

    torch.save(gru.state_dict(), path_gru)

In [15]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))
# gru.to(torch.device('cpu')).eval()

<All keys matched successfully>

# ANALYSIS

In [16]:
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals

In [17]:
estimator.eval()
evc.eval()
gru.eval()

GRUModel(
  (participant_embedding): Embedding(1, 16)
  (experiment_embedding): Embedding(1, 16)
  (linear_in): Linear(in_features=6, out_features=16, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (gru): GRU(16, 16, batch_first=True)
  (linear_out): Linear(in_features=16, out_features=2, bias=True)
)

## Analysis model evaluation

In [18]:
print("Model evaluation on training data (for AIC and BIC): ")
analysis_model_evaluation(
    dataset=dataset_train,
    spice_model=estimator,
    benchmark_model=evc,
    gru_model=gru,
)

Model evaluation on training data (for AIC and BIC): 
Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


W0611 12:19:33.017000 19818 site-packages/torch/_inductor/utils.py:1250] [0/5] Not enough SMs to use max_autotune_gemm mode
W0611 12:19:33.148000 19818 site-packages/torch/utils/_sympy/interp.py:176] [0/5] failed while executing pow_by_natural([VR[3, int_oo], VR[-1, -1]])


,Trial Lik.,(std),n_parameters,(std),NLL,AIC,BIC
Benchmark,0.570419,0.155806,5.000000,0.000000,20643.751953,41297.503906,41340.066406
GRU,0.696628,0.152761,1810.000000,0.000000,13293.589844,30207.179688,45614.835938
SPICE-RNN,0.667695,0.149850,12368.000000,0.000000,14853.480469,54442.960938,159725.796875
SPICE-SYM,0.554218,0.133504,11.095239,0.797461,21703.312500,43428.816406,43523.265625


In [19]:
print("Model evaluation on held-out data (for average trial likelihood and NLL): ")
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    benchmark_model=evc,
    gru_model=gru,
)

Model evaluation on held-out data (for average trial likelihood and NLL): 
Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


,Trial Lik.,(std),n_parameters,(std),NLL,AIC,BIC
Benchmark,0.546326,0.159768,5.000000,0.000000,7515.628906,15041.257812,15078.398438
GRU,0.662498,0.155989,1810.000000,0.000000,5118.724609,13857.449219,27302.181641
SPICE-RNN,0.653976,0.147707,12368.000000,0.000000,5279.683594,35295.367188,127165.234375
SPICE-SYM,0.546696,0.131188,11.095239,0.797461,7507.224609,15036.639648,15119.055664


## Analysis generative behavior

In [ ]:
generated_dataset_spice = generate_behavior(
    dataset=dataset_train,
    model=estimator,
    save_dataset='data/braun2018_spice.csv'
)

generated_dataset_benchmark = generate_behavior(
    dataset=dataset_train,
    model=evc,
    save_dataset='data/braun2018_benchmark.csv'
)

generated_dataset_gru = generate_behavior(
    dataset=dataset_train,
    model=gru,
    save_dataset='data/braun2018_gru.csv'
)

In [ ]:
from analysis_generative import analysis_generative_behavior

In [ ]:
analysis_generative_behavior(
    path_data_real='data/braun2018.csv',
    path_data_benchmark='data/braun2018_benchmark.csv',
    path_data_gru='data/braun2018_gru.csv',
    path_data_spice_rnn='data/braun2018_spice.csv',
    path_data_spice='data/braun2018_spice.csv',
)

## Analysis coefficient distribution

In [ ]:
analysis_coefficients_distributions(
    spice_model=estimator,
    output_dir='results',
)

## Analysis coefficients individuals